# Evaluate all runs

Loads every experiment folder under `training_runs/`, evaluates on val/test, fits optional temperature scaling, and writes plots/metrics into each run’s `eval_plots`/`eval_metrics.json`.

In [ ]:

from __future__ import annotations

from pathlib import Path
import sys

import json
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from typing import Dict
from sklearn.metrics import confusion_matrix

sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "legend.fontsize": 10,
})


here = Path.cwd().resolve()
for candidate in [here] + list(here.parents):
    src_dir = candidate / "src"
    if src_dir.exists():
        src_str = str(src_dir)
        if src_str not in sys.path:
            sys.path.insert(0, src_str)
        break

from utils import resolve_project_root

PROJECT_ROOT = resolve_project_root()
project_src = PROJECT_ROOT / "src"
if str(project_src) not in sys.path:
    sys.path.insert(0, str(project_src))


from config import TrainConfig
from data.datamodule import DataModule
from models.builder import build_model
from validation.calibration import TemperatureScaler
from validation.evaluate import evaluate
from validation.metrics import reliability_bins

In [ ]:
RUNS_ROOT = PROJECT_ROOT / "training_runs"
CALIBRATE = True  # False if calibration is already applied at training time

In [3]:
def set_plot_dir(run_dir: Path):
    plots = run_dir / "eval_plots"
    plots.mkdir(parents=True, exist_ok=True)
    return plots

def save_fig(fig, plots_dir: Path, name: str):
    path = plots_dir / f"{name}.pdf"
    fig.savefig(path, bbox_inches="tight", format="pdf")
    print(f"Saved {path}")
    plt.close(fig)

def plot_metric_bars(split: str, tag: str, metrics: Dict, model_name: str, plots_dir: Path):
    keys = [
        ("accuracy", "Accuracy"),
        ("macro_precision", "Macro Precision"),
        ("macro_recall", "Macro Recall"),
        ("macro_f1", "Macro F1"),
        ("brier", "Brier"),
        ("ece", "ECE"),
    ]
    labels = [label for key, label in keys if key in metrics]
    values = [metrics[key] for key, _ in keys if key in metrics]
    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    bars = ax.bar(labels, values, color=plt.get_cmap("tab10")(range(len(values))))
    ax.set_ylabel("Score")
    ax.set_ylim(bottom=0)
    ax.set_title(f"{model_name} | {split} ({tag}) metrics")
    ax.grid(axis="y", alpha=0.25)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, val, f"{val:.3f}", ha="center", va="bottom", fontsize=9)
    plt.xticks(rotation=20)
    plt.tight_layout()
    save_fig(fig, plots_dir, f"{split}_{tag}_metrics")

def plot_outputs(split: str, tag: str, outputs: Dict, class_names, cfg: TrainConfig, model_name: str, plots_dir: Path):
    probs = outputs["probs"]
    labels = outputs["labels"].cpu().numpy()
    preds = probs.argmax(axis=1)

    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"{model_name} | {split} Confusion ({tag})")
    fig.tight_layout()
    save_fig(fig, plots_dir, f"{split}_{tag}_confusion")

    pos_scores = probs[:, 1] if probs.shape[1] > 1 else probs[:, 0]
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for label_val, label_name in enumerate(class_names):
        ax.hist(pos_scores[labels == label_val], bins=20, alpha=0.6, label=label_name, density=True)
    ax.set_xlabel("Predicted probability (class 1)")
    ax.set_ylabel("Density")
    ax.set_title(f"{model_name} | {split} Score Histogram ({tag})")
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()
    save_fig(fig, plots_dir, f"{split}_{tag}_score_hist")

    bin_conf, bin_acc, bin_count = reliability_bins(probs, labels, n_bins=cfg.reliability_bins)
    fig, ax = plt.subplots(figsize=(6.5, 5))
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect calibration")
    ax.bar(bin_conf, bin_acc, width=1.0 / cfg.reliability_bins, alpha=0.65, align="center", label="Observed")
    ax.set_xlabel("Confidence")
    ax.set_ylabel("Accuracy")
    ax.set_title(f"{model_name} | {split} Reliability ({tag})")
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()
    save_fig(fig, plots_dir, f"{split}_{tag}_reliability")

def load_config(run_dir: Path) -> TrainConfig:
    cfg_dict = json.loads((run_dir / "config.json").read_text())
    return TrainConfig(**cfg_dict)

def load_model(cfg: TrainConfig, ckpt_path: Path):
    model = build_model(cfg)
    state = torch.load(ckpt_path, map_location="cpu")
    model.load_state_dict(state["model_state"])
    return model

def build_loaders(cfg: TrainConfig):
    # Ensure notebook-friendly workers
    cfg.num_workers = 0
    datamodule = DataModule(cfg)
    datamodule.setup()
    return datamodule.val_dataloader(), datamodule.test_dataloader(), datamodule.train_dataset.classes

def evaluate_run(run_dir: Path):
    print(f"\n=== {run_dir.name} ===")
    cfg = load_config(run_dir)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    plots_dir = set_plot_dir(run_dir)

    val_loader, test_loader, class_names = build_loaders(cfg)
    ckpt_path = run_dir / "artifacts" / "checkpoints" / "best.pt"
    if not ckpt_path.exists():
        print("No checkpoint found; skipping.")
        return
    model = load_model(cfg, ckpt_path).to(device)

    val_metrics_raw, val_outputs_raw = evaluate(model, val_loader, device, cfg, temperature=None)
    test_metrics_raw, test_outputs_raw = evaluate(model, test_loader, device, cfg, temperature=None)

    plot_metric_bars("val", "raw", val_metrics_raw, cfg.model_name, plots_dir)
    plot_outputs("val", "raw", val_outputs_raw, class_names, cfg, cfg.model_name, plots_dir)
    plot_metric_bars("test", "raw", test_metrics_raw, cfg.model_name, plots_dir)
    plot_outputs("test", "raw", test_outputs_raw, class_names, cfg, cfg.model_name, plots_dir)

    eval_metrics = {
        "val_raw": val_metrics_raw,
        "test_raw": test_metrics_raw,
    }

    if CALIBRATE:
        temperature = TemperatureScaler().to(device)
        temperature.fit(val_outputs_raw["logits"].to(device), val_outputs_raw["labels"].to(device))
        temp_value = float(temperature.temperature.item())

        val_metrics_cal, val_outputs_cal = evaluate(model, val_loader, device, cfg, temperature=temperature)
        test_metrics_cal, test_outputs_cal = evaluate(model, test_loader, device, cfg, temperature=temperature)

        plot_metric_bars("val", "calibrated", val_metrics_cal, cfg.model_name, plots_dir)
        plot_outputs("val", "calibrated", val_outputs_cal, class_names, cfg, cfg.model_name, plots_dir)
        plot_metric_bars("test", "calibrated", test_metrics_cal, cfg.model_name, plots_dir)
        plot_outputs("test", "calibrated", test_outputs_cal, class_names, cfg, cfg.model_name, plots_dir)

        (run_dir / "artifacts" / "temperature_eval.pt").parent.mkdir(parents=True, exist_ok=True)
        temperature.save(str(run_dir / "artifacts" / "temperature_eval.pt"))
        eval_metrics.update({
            "val_calibrated": val_metrics_cal,
            "test_calibrated": test_metrics_cal,
            "temperature": temp_value,
        })

    (run_dir / "eval_metrics.json").write_text(json.dumps(eval_metrics, indent=2))
    print("Saved metrics.")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

run_dirs = [p for p in RUNS_ROOT.iterdir() if (p / "config.json").exists()]
for run_dir in sorted(run_dirs):
    evaluate_run(run_dir)
print("Done.")



=== data_2025_12_11-22_35_convnext_small ===
Saved /home/tim/repos/ipeo-hurricane-damage-detection/training_runs/data_2025_12_11-22_35_convnext_small/eval_plots/val_raw_metrics.pdf
Saved /home/tim/repos/ipeo-hurricane-damage-detection/training_runs/data_2025_12_11-22_35_convnext_small/eval_plots/val_raw_confusion.pdf
Saved /home/tim/repos/ipeo-hurricane-damage-detection/training_runs/data_2025_12_11-22_35_convnext_small/eval_plots/val_raw_score_hist.pdf
Saved /home/tim/repos/ipeo-hurricane-damage-detection/training_runs/data_2025_12_11-22_35_convnext_small/eval_plots/val_raw_reliability.pdf
Saved /home/tim/repos/ipeo-hurricane-damage-detection/training_runs/data_2025_12_11-22_35_convnext_small/eval_plots/test_raw_metrics.pdf
Saved /home/tim/repos/ipeo-hurricane-damage-detection/training_runs/data_2025_12_11-22_35_convnext_small/eval_plots/test_raw_confusion.pdf
Saved /home/tim/repos/ipeo-hurricane-damage-detection/training_runs/data_2025_12_11-22_35_convnext_small/eval_plots/test_raw_